# Coalition Finding

Let's assume we are explanining a prediction of a model with a set of features $N$ using some value function $\nu: \mathcal{P}(N) \rightarrow \R$.
We obtain an explanation vector $e = \{e_T\}_{T\subseteq N, |T| \leq k}$ which assigns an interaction value $e_T$ to all coalitions $T$ up to some order $k \in \N$.
From this explanation vector, we can create a _simplified game_ $\hat \nu_e$ which approximates the original value function $\nu$:
$$
    \hat \nu_e(S) := \sum_{T\subseteq S, |T|\leq k} e_T
$$

$\def\min{\text{min}}
\def\max{\text{max}}$
Our goal will now be to find for some size $\ell \in \N$ the coalitions $S_\ell^\min$ and $S_\ell^\max$ that minimize and maximize $\hat \nu_e$ respectively.
An exhaustive search over all $S \subseteq N$ would have an exponential runtime complexity and is therefore not feasible for large feature sets $N$.
Instead, we turn to heuristic approaches in order to approximate $S_\ell^\min$ and $S_\ell^\max$.

## Strategy No. 1: Solos

Our first strategy is called 'Solos'.
As the name suggests, every player is considered individually: We rank all players $p \in N$ by looking only at their Shapley value $e_{\{p\}}$, ignoring all interactions of order $k \geq 2$.
We then choose the $\ell$ players with the lowest Shapley values as the minimal coalition $S_\ell^\min$ and the $\ell$ players with the highest values as $S_\ell^\max$.

This makes for a runtime of $O(n \log n)$, where $n := |N|$, since we only need to sort the array of Shapley values in order to identify the $\ell$ highest and lowest players.
(This assumes, of course, that the interaction values vector $e$ is represented in memory in such a way that all interactions of order $1$ can be retrieved without having to traverse its entirety.)

## Strategy No. 2: Equal Payoff

Our second strategy, called 'Equal Payoff', is slightly more sophisticated than the first one.
Instead of only considering Shapley values, we take all Shapley interactions into account.
We assign a score $s_p$ to each player $p$ by distributing the value of all interactions evenly among their respective participants.
(The baseline $e_\emptyset$ can simply be ignored since it doesn't affect $S_\ell^\min$ and $S_\ell^\max$ anyway.)
The score $s_p$ is defined as follows:
$$
    s_p := \sum_{T\subseteq S,\> p \in T} \frac{e_T}{|T|}
$$

After computing $s_p$ for all players $p \in \N$, we again sort them by their scores and choose the best and worst $\ell$ players as $S_\ell^\max$ and $S_\ell^\min$.

To compute the scores, we iterate over all subsets $T$ of order $\leq k$ and each time increase $s_p$ for all $p \in T$. Therefore the amount of operations required is
$$
    \sum_{j=1}^k \> j \binom{n}{j}
    \leq \sum_{j=1}^k \> j \cdot n^j.
$$
The complexity is decided by the largest exponent of $n$, giving us $O(n^k)$.

## Strategy No. 3: Greedy Search

This strategy is quite simple. We start by choosing the player with the lowest (respectively highest) Shapley value, then keep adding new players until we reach the desired coalition size $\ell$, each time choosing the player that increase the total value of the coalition the least (the most).

By precomputing a mapping from each player to the coalitions he's part of, we can compute the increase in payoff for a potential new player with $k \cdot n^k$ operations. This check needs to be performed for (almost) all $n$ potential new joiners in every itertaion until we've reached a size of $\ell$ players, giving us a final complexity of $O(\ell k n^{k+1}) = O(n^{k+1})$.

## Example

In this example, we'll generate a Sum of Unanimity games, which is the linear sum of a bunch of _Unanimity_ subgames. Each of the subgames is defined by a subset $U \subseteq N$ and has utility 1 iff all players of $U$ are present in the coalition.
From that, we generate an explanation using the `ExactComputer`.

In [ ]:
from shapiq import ExactComputer
from shapiq.games.benchmark import SOUM

n_players = 10
explanation_order = 3
game = SOUM(n=10, n_basis_games=50, random_state=42)
computer = ExactComputer(n_players=game.n_players, game=game)
iv = computer(index="FSII", order=explanation_order)
iv

Now, we can run our coalition finding algorithms on the explanation.